# Step 3 – Nepal September 2024 flood: Sentinel-1 data preparation
For two study areas (**Kathmandu Valley** = steep/urban, **Koshi Terai** = flat comparison) this notebook:
1. Lists every Sentinel-1 acquisition around the flood (27–29 Sep 2024) and picks the flood and pre-flood images.
2. Shows how cloudy Sentinel-2 was (evidence for why SAR is needed).
3. Builds a stack per area: flood VV/VH, pre-flood VV/VH, slope, HAND, permanent water, layover/shadow mask, and a quick change-detection flood map.
4. Exports each stack as a GeoTIFF (10 m, UTM 45N) to Google Drive.

No GPU needed: *Runtime → Change runtime type → CPU* is fine.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID

import ee, math, time, pandas as pd, geemap
ee.Authenticate(); ee.Initialize(project=PROJECT)

# Study areas – edit the boxes if needed: [west, south, east, north]
AOIS = {
    'kathmandu': ee.Geometry.Rectangle([85.20, 27.58, 85.48, 27.78]),   # Bagmati, Kathmandu Valley
    'koshi_terai': ee.Geometry.Rectangle([86.85, 26.45, 87.15, 26.75]), # Koshi barrage / Sunsari plains
}
FLOOD_START, FLOOD_END = '2024-09-27', '2024-10-04'   # window to search for the flood image
SEARCH_START, SEARCH_END = '2024-08-15', '2024-10-20'  # window for the scene table
print('Earth Engine OK')

## 1. Which Sentinel-1 images exist?
In 2024 only Sentinel-1A was operating (12-day repeat), so coverage near the flood peak may be thin. `cover` = fraction of the study area inside the scene.

In [ ]:
def s1(aoi):
    return (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))

def scene_table(aoi):
    col = s1(aoi).filterDate(SEARCH_START, SEARCH_END).map(
        lambda im: im.set('cover', im.geometry().intersection(aoi, 1).area(1).divide(aoi.area(1))))
    rows = col.reduceColumns(ee.Reducer.toList(4),
        ['system:time_start', 'orbitProperties_pass', 'relativeOrbitNumber_start', 'cover']).get('list').getInfo()
    df = pd.DataFrame(rows, columns=['t', 'pass', 'orbit', 'cover'])
    df['date'] = pd.to_datetime(df.t, unit='ms').dt.strftime('%Y-%m-%d')
    # same day + same orbit = one acquisition (several frames)
    return (df.groupby(['date', 'pass', 'orbit'], as_index=False)['cover'].sum()
              .assign(cover=lambda d: d.cover.clip(upper=1).round(2)).sort_values('date'))

tables = {name: scene_table(aoi) for name, aoi in AOIS.items()}
for name, t in tables.items():
    print(f'\n=== {name} ==='); display(t)

## 2. Pick flood and pre-flood images
Flood image = first acquisition in the flood window covering ≥ 60 % of the area. Pre-flood = latest earlier acquisition on the **same orbit** (same viewing geometry, so the change is real).

To choose differently, set `OVERRIDE`, e.g. `{'kathmandu': ('2024-09-30', '2024-09-18', 85)}` as (flood date, pre date, orbit).

In [ ]:
OVERRIDE = {}

def pick(t):
    f = t[(t.date >= FLOOD_START) & (t.date <= FLOOD_END) & (t.cover >= 0.6)]
    if f.empty: return None
    f = f.iloc[0]
    p = t[(t.orbit == f.orbit) & (t.date < '2024-09-25') & (t.cover >= 0.6)]
    if p.empty: return None
    return f.date, p.iloc[-1].date, int(f.orbit)

choice = {}
for name, t in tables.items():
    c = OVERRIDE.get(name) or pick(t)
    if c is None:
        print(f'⚠️ {name}: no usable flood/pre pair in the window – look at the table above and set OVERRIDE')
    else:
        choice[name] = c; print(f'{name}: flood {c[0]}, pre-flood {c[1]}, orbit {c[2]}')

## 3. How cloudy was Sentinel-2? (motivation for SAR)

In [ ]:
for name, aoi in AOIS.items():
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi).filterDate('2024-09-20', '2024-10-10')
    r = s2.reduceColumns(ee.Reducer.toList(2), ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE']).get('list').getInfo()
    d = pd.DataFrame(r, columns=['t', 'cloud %'])
    d['date'] = pd.to_datetime(d.t, unit='ms').dt.strftime('%Y-%m-%d')
    print(f'\n=== {name} ==='); display(d.groupby('date')['cloud %'].mean().round(0).to_frame())

## 4. Build the stacks
Bands: `flood_VV, flood_VH, pre_VV, pre_VH, slope, hand, perm_water, ls_mask, quick_flood`
- **ls_mask:** 1 = layover, 2 = shadow (approximate, from DEM + look direction + incidence angle).
- **quick_flood:** classic change detection (VV < −15 dB and drop > 3 dB, not permanent water). A non-AI reference baseline for the paper.

In [ ]:
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic().setDefaultProjection('EPSG:4326', None, 30)
slope = ee.Terrain.slope(dem); aspect = ee.Terrain.aspect(dem)
hand = ee.Image('MERIT/Hydro/v1_0_1').select('hnd').unmask(0)
perm = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').gte(80).unmask(0)
DEG = math.pi / 180

def acq(aoi, date, orbit):
    d = ee.Date(date)
    return (s1(aoi).filterDate(d, d.advance(1, 'day'))
            .filter(ee.Filter.eq('relativeOrbitNumber_start', orbit))
            .select(['VV', 'VH', 'angle']).mosaic().clip(aoi))

def layover_shadow(img, aoi):
    theta = img.select('angle')
    look = ee.Number(ee.Terrain.aspect(theta).reduceRegion(ee.Reducer.mean(), aoi, 1000).get('aspect'))
    alpha = slope.multiply(DEG).tan().multiply(aspect.subtract(look).multiply(DEG).cos()).atan().divide(DEG)
    shadow = alpha.gt(ee.Image(90).subtract(theta))
    layover = alpha.multiply(-1).gt(theta)
    return layover.add(shadow.multiply(2)).rename('ls_mask')

stacks = {}
for name, (fd, pd_, orb) in choice.items():
    aoi = AOIS[name]; f, p = acq(aoi, fd, orb), acq(aoi, pd_, orb)
    diff = f.select('VV').subtract(p.select('VV'))
    quick = f.select('VV').lt(-15).And(diff.lt(-3)).And(perm.Not()).rename('quick_flood')
    stacks[name] = ee.Image.cat([
        f.select(['VV', 'VH'], ['flood_VV', 'flood_VH']),
        p.select(['VV', 'VH'], ['pre_VV', 'pre_VH']),
        slope.rename('slope'), hand.rename('hand'), perm.rename('perm_water'),
        layover_shadow(f, aoi), quick]).toFloat().clip(aoi)
print('stacks:', list(stacks))

## 5. Look at it on a map

In [ ]:
name = list(stacks)[0]          # change to 'koshi_terai' to view the other area
st = stacks[name]
m = geemap.Map(); m.centerObject(AOIS[name], 12)
m.addLayer(st.select('pre_VV'), {'min': -25, 'max': 0}, 'pre-flood VV')
m.addLayer(st.select('flood_VV'), {'min': -25, 'max': 0}, 'flood VV')
m.addLayer(st.select('ls_mask').selfMask(), {'min': 1, 'max': 2, 'palette': ['orange', 'black']}, 'layover / shadow')
m.addLayer(st.select('quick_flood').selfMask(), {'palette': ['00a2ff']}, 'quick flood map')
m

## 6. Export to Google Drive
Files go to the Drive of the **Earth Engine account**, folder `Paper2_flood_nepal`. Takes ~5–20 min.

In [ ]:
tasks = []
for name, st in stacks.items():
    fd, pd_, orb = choice[name]
    t = ee.batch.Export.image.toDrive(image=st, description=f'nepal_{name}',
            folder='Paper2_flood_nepal', fileNamePrefix=f'{name}_flood{fd}_pre{pd_}_orbit{orb}',
            region=AOIS[name], scale=10, crs='EPSG:32645', maxPixels=1e10)
    t.start(); tasks.append(t)

while any(t.status()['state'] in ('READY', 'RUNNING') for t in tasks):
    print(' | '.join(f"{t.status()['description']}: {t.status()['state']}" for t in tasks)); time.sleep(30)
for t in tasks: print(t.status()['description'], t.status()['state'], t.status().get('error_message', ''))